In [ ]:
import getpass
import os

try:
    from dotenv import load_dotenv
    # 加载环境变量
    load_dotenv()
except ImportError:
    # 如果 python-dotenv 未安装，则定义一个空函数作为回退
    def load_dotenv():
        print("load_dotenv not found")
        pass

def check_smith():
    os.environ["LANGSMITH_TRACING"] = "true"
    if "LANGSMITH_API_KEY" not in os.environ:
        os.environ["LANGSMITH_API_KEY"] = getpass.getpass(
            prompt="Enter your LangSmith API key (optional): "
        )
    if "LANGSMITH_PROJECT" not in os.environ:
        os.environ["LANGSMITH_PROJECT"] = getpass.getpass(
            prompt='Enter your LangSmith Project Name (default = "default"): '
        )
        if not os.environ.get("LANGSMITH_PROJECT"):
            os.environ["LANGSMITH_PROJECT"] = "default"
    print("check smith OK")


def check_openai_key():
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")
    print("check openai key OK")

def check_tavily_key():
    if not os.environ.get("TAVILY_API_KEY"):
        os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter API key for TAVILY: ")
    print("check tavily key OK")

check_smith()
check_openai_key()
check_tavily_key()

In [ ]:
# Import relevant functionality
from langchain_tavily import TavilySearch

search = TavilySearch(max_results=2)
search_results = search.invoke("What is the weather in Shanghai")
print(search_results)

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

model = init_chat_model("gpt-4.1-nano", model_provider="openai")
query = "Hi!"
response = model.invoke([{"role": "user", "content": query}])
print(response.text())

In [ ]:

search = TavilySearch(max_results=2)
tools = [search]
# create_react_agent 内部调用.bind_tools方法
agent_executor = create_react_agent(model, tools)

input_message = {"role": "user", "content": "Hi!"}
response = agent_executor.invoke({"messages": [input_message]})

for message in response["messages"]:
    message.pretty_print()

In [ ]:
# Create the agent
memory = MemorySaver()
agent_executor = create_react_agent(model, tools, checkpointer=memory)
# Use the agent
config = {"configurable": {"thread_id": "abc123"}}

input_message = {
    "role": "user",
    "content": "Hi, 我是Bob, 我生活在上海.",
}
# 流输出
for step in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

In [ ]:
input_message = {
    "role": "user",
    "content": "我生活的地方天气怎么样?",
}
for step in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

In [ ]:

for step, metadata in agent_executor.stream(
    {"messages": [input_message]}, config=config, stream_mode="messages"
):
    if metadata["langgraph_node"] == "agent" and (text := step.text()):
        print(text, end="|")

In [ ]:
agent_executor = create_react_agent(model, [TavilySearch(max_results=2)], checkpointer=memory)
input_message = {
    "role": "user",
    "content": "上海的天气",
}
config = {"configurable": {"thread_id": "xyz123"}}
# python3.11也可使用.astream_events流回token令牌
async for event in agent_executor.astream_events(
    {"messages": [input_message]},config=config
):
    kind = event["event"]
    if kind == "on_chain_start":
        if event["name"] == "Agent":
        # Was assigned when creating the agent with `.with_config({"run_name": "Agent"})`
            print(f"Starting agent: {event['name']} with input: {event['data'].get('input')}")
    elif kind == "on_chain_end":
        if event["name"] == "Agent":
            # Was assigned when creating the agent with `.with_config({"run_name": "Agent"})`
            print()
            print("--")
            print(f"Done agent: {event['name']} with output: {event['data'].get('output')['output']}")
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")
    elif kind == "on_tool_start":
        print("--")
        print(f"Starting tool: {event['name']} with inputs: {event['data'].get('input')}")
    elif kind == "on_tool_end":
        print(f"Done tool: {event['name']}")
        print(f"Tool output was: {event['data'].get('output')}")
        print("--")

In [ ]:
# 绑定工具
model_with_tools = model.bind_tools(tools)
query = "Hi!"
response = model_with_tools.invoke([{"role": "user", "content": query}])
# 此处工具不会调用
print(f"Message content: {response.text()}")
print(f"Tool calls: {response.tool_calls}\n")

query = "Search for the weather in SF"
response = model_with_tools.invoke([{"role": "user", "content": query}])
# 此处将调用工具
print(f"Message content: {response.text()}\n")
print(f"Tool calls: {response.tool_calls}")